# ==============================================================================
# 🎬 LARGE-SCALE SENTIMENT ANALYSIS: IMDB REVIEWS VIA MULTINOMIAL NAIVE BAYES
# ==============================================================================
### Architectural Overview:
Analyzing 50,000 long-form movie reviews with Multinomial Naive Bayes.
Multinomial Naive Bayes computes relative log probabilities:
$$\log P(y = 1 \mid X) - \log P(y = 0 \mid X) = \log \frac{P(y=1)}{P(y=0)} + \sum_{i=1}^d x_i \log \frac{p_{1i}}{p_{0i}}$$

Using `sublinear_tf=True` dampens the impact of words repeated excessively in verbose film reviews.


In [1]:
# STEP 1: IMPORTS & ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Sentiment Analysis Environment initialized.")


✅ STEP 1: Sentiment Analysis Environment initialized.


## 📥 STEP 2 & 3: INGESTION & SANITIZATION
Loading the Kaggle IMDB 50k Movie Reviews dataset.


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/movie_reviews/IMDB Dataset.csv'
df = pd.read_csv(data_path)

# Map sentiment to binary integer (positive=1, negative=0)
df['sentiment_binary'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Sample 15,000 reviews for efficient enterprise training while preserving statistical power
df_sample = df.sample(n=15000, random_state=42).reset_index(drop=True)

print(f"📊 Sampled Review Corpus: {df_sample.shape[0]:,} reviews")
print(f"Class Balance: {df_sample['sentiment_binary'].value_counts().to_dict()}")
df_sample.head(3)


📊 Sampled Review Corpus: 15,000 reviews
Class Balance: {1: 7589, 0: 7411}


,review,sentiment,sentiment_binary
0,I really liked this Summerslam due to the look...,positive,1
1,Not many television shows appeal to quite as m...,positive,1
2,The film quickly gets to a major chase scene w...,negative,0


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating text and target, auditing missing reviews.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df_sample['review']
y = df_sample['sentiment_binary']

print(f"Missing review texts: {X.isnull().sum()}")
print(f"Target Distribution:\n{y.value_counts(normalize=True).round(4) * 100}")


Missing review texts: 0
Target Distribution:
sentiment_binary
1    50.59
0    49.41
Name: proportion, dtype: float64


## 🔒 STEP 6: STRATIFIED SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Partition : {len(X_train):,} reviews")
print(f"Test Partition  : {len(X_test):,} reviews")


Train Partition : 12,000 reviews
Test Partition  : 3,000 reviews


## ⚙️ STEP 7: SUBLINEAR TF-IDF VECTORIZATION
Configuring `TfidfVectorizer` with 15,000 features and unigrams + bigrams.


In [5]:
# STEP 7: VECTORIZER
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=15000,
    sublinear_tf=True
)
print("✅ STEP 7: Bigram TF-IDF sentiment pipeline ready.")


✅ STEP 7: Bigram TF-IDF sentiment pipeline ready.


## ⚠️ STEP 8: BASELINE PIPELINE
Training baseline MultinomialNB with default $\alpha=1.0$.


In [6]:
# STEP 8: BASELINE FIT
pipe_base = Pipeline([
    ('tfidf', tfidf),
    ('mnb', MultinomialNB(alpha=1.0))
])
pipe_base.fit(X_train, y_train)

y_pred_base = pipe_base.predict(X_test)
print(f"Baseline MultinomialNB Accuracy: {accuracy_score(y_test, y_pred_base)*100:.2f}%")
print(f"Baseline MultinomialNB ROC-AUC : {roc_auc_score(y_test, pipe_base.predict_proba(X_test)[:, 1])*100:.2f}%")


Baseline MultinomialNB Accuracy: 85.37%


Baseline MultinomialNB ROC-AUC : 93.24%


## 🎯 STEP 9: TUNING $\alpha$
Optimizing Laplace smoothing across 15 logarithmic values.


In [7]:
# STEP 9: HYPERPARAMETER TUNING
param_grid = {
    'mnb__alpha': np.logspace(-3, 1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipe_base,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Alpha Parameter: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Alpha Parameter: {'mnb__alpha': np.float64(2.6826957952797246)}
🎯 Best 5-Fold CV ROC-AUC: 93.71%


## ⚔️ STEP 10: ALGORITHM DUEL (MultinomialNB vs Logistic Regression)
Evaluating MultinomialNB against Logistic Regression.


In [8]:
# STEP 10: MODEL BENCHMARK
pipe_lr = Pipeline([
    ('tfidf', tfidf),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

y_pred_nb = best_model.predict(X_test)
y_pred_lr = pipe_lr.predict(X_test)

print("="*65)
print(f"MultinomialNB (Tuned) -> Test Accuracy: {accuracy_score(y_test, y_pred_nb)*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])*100:.2f}%")
print(f"Logistic Regression   -> Test Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:, 1])*100:.2f}%")
print("="*65)


MultinomialNB (Tuned) -> Test Accuracy: 85.50%, ROC-AUC: 93.31%


Logistic Regression   -> Test Accuracy: 87.73%, ROC-AUC: 94.99%


## 📊 STEP 11: EVALUATION REPORT & CONFUSION MATRIX
Review classification performance breakdown.


In [9]:
# STEP 11: EVALUATION
print("📋 IMDB SENTIMENT CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_nb, target_names=['Negative Review', 'Positive Review']))

cm = confusion_matrix(y_test, y_pred_nb)
print(f"Confusion Matrix:\n{cm}")


📋 IMDB SENTIMENT CLASSIFICATION REPORT:
                 precision    recall  f1-score   support

Negative Review       0.86      0.84      0.85      1482
Positive Review       0.85      0.87      0.86      1518

       accuracy                           0.85      3000
      macro avg       0.86      0.85      0.85      3000
   weighted avg       0.86      0.85      0.85      3000

Confusion Matrix:
[[1241  241]
 [ 194 1324]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Exporting pipeline and validating live sentiment latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/imdb_sentiment_multinomial_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_critique = [
    "A breathtaking cinematic masterpiece with stunning visuals, gripping narrative, and flawless acting performances throughout."
]

t0 = time.perf_counter()
pred_sentiment = loaded_pipe.predict(sample_critique)[0]
sentiment_prob = loaded_pipe.predict_proba(sample_critique)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🎬 LIVE SENTIMENT SMOKE TEST:")
print(f"   Predicted Sentiment : {'🌟 POSITIVE' if pred_sentiment == 1 else '💔 NEGATIVE'}")
print(f"   Positivity Confidence: {sentiment_prob*100:.2f}%")
print(f"   Inference Latency   : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/imdb_sentiment_multinomial_nb_pipeline.joblib (1,052,145 bytes)



🎬 LIVE SENTIMENT SMOKE TEST:
   Predicted Sentiment : 🌟 POSITIVE
   Positivity Confidence: 82.40%
   Inference Latency   : 2.685 ms (SLA < 2.0ms: CHECK)
